# Lesson 4: Extracting Toponyms in Texts

## Overview

Up till now, we have not done anything with the data that is particularly complex. We have found major keywords and trend lines in posting. We don't know that much more about the data. In order to do this, we have to use more complex language models to get a better sense of the "aboutness" of the text. In particular, we are interested in what locations JMU students talk about and how they talk about them as compared to other students. For this we will use more sophisticated packages that contain more complex language models. This lesson will cover Named Entity Recognition models. These models recognize particular types of words like locations.

This lesson will cover three sections:

- Prepping the data
- Running NER
- Visualizing the data



## Introduction

In this lesson we will learn how to extract place names from text. We will do so in two ways:

- Using known place names
- Natural Language Processing

Natural Language Processing (NLP) teaches computers how to "read" and understand text like humans do. When we want to find place names in text, this becomes tricky because the same word can mean different things depending on how it's used.

For example: 

- I visited **Washington** last summer and saw the Capitol building.
- **Washington** led the Continental Army during the Revolutionary War.

In the first sentence, "Washington" refers to Washington D.C. (a place). In the second sentence, "Washington" refers to George Washington (a person, not a place). Humans can easily tell the difference, but computers need help figuring this out.

If we had a system where we simply gave computers a list of place names, we would get a lot of false positives, cases where the computer thinks it is true but it is actually false. Therefore linguists started teaching computers the rules of grammar so they could understand **parts of speech**. By knowing *how* a word is being used, computers get more accurate in predicting what the word means in that context. 

By having a basic grammar it is easier to figure out what a word means:

- The weather in London was cold and foggy.
- Jack London wrote *Call of the Wild*.

Here, a computer can figure out that the first "London" is a place (because we say "in London"), while "Jack London" is a person's name because it is performing the verb "write", something cities generally don't do.

Over time, more sophisticated models have developed to help computers understand how language is being used. This process of finding locations, people, organizations, and other important information in text is called Named Entity Recognition (NER). It's very useful because it helps us understand what a text is really about, beyond just counting words.

## 1 Load Libraries and Data

Most Python scripts follow the same three-step opening pattern. Recognising it will help you read unfamiliar code much more quickly:

| Step | What it does | Code pattern |
|---|---|---|
| **Load libraries** | Import the tools you need | `import pandas as pd` |
| **Load data** | Read the dataset into memory | `pd.read_pickle(...)` |
| **Verify data** | Confirm the data looks right before processing | `df.sample(...)` |

This lesson uses two libraries for natural language processing:

1. `spacy` — An industrial-strength NLP library that uses pre-trained language models to parse text, assign grammatical roles to words, and identify **named entities** such as people, organizations, and places (GPE — Geopolitical Entities). We use it to extract location names from Reddit posts.
2. `nltk` — Natural Language Toolkit: a broad collection of NLP utilities. In this lesson we use its sentence tokenizer to split post text into individual sentences before running entity recognition.

In [3]:
# ── Step 1: Load libraries ────────────────────────────────────────────────────
# All imports go at the top so dependencies are visible at a glance.
import pandas as pd   # data tables
import nltk           # sentence tokenizer
import re             # regular expressions (used later for text cleaning)

# ── Step 2: Load data ─────────────────────────────────────────────────────────
# Read the cleaned Reddit DataFrame from the pickle file produced in Lesson 3.
# Pickle preserves column types (category, datetime, etc.) set in that lesson.
df_reddit = pd.read_pickle("data/jmu_reddit.pickle")

# ── Step 3: Verify data ───────────────────────────────────────────────────────
# Always inspect the data before processing.
# .sample() shows random rows rather than just the first ones (.head()),
# which helps catch patterns that only appear further into the dataset.
# random_state=43 makes the sample repeatable — the same random rows will always appear.
df_reddit.sample(n=5, random_state=43)

,type,title,text,date,score,year_month
2238,comment,Explosion?,I saw it firsthand drove by it,2020-10-17 17:11:22,2,2020-10
8929,comment,Frustrated Student,"Oh I absolutely agree. But these ""kids"" also n...",2020-10-04 19:10:22,3,2020-10
6629,comment,JMU President,"Like don’t get me wrong, I liked Alger. Grante...",2025-08-30 21:15:12,2,2025-08
412,comment,Consider: the Duke Dog with no eyebrows,Homie is vibing hard,2019-11-29 01:30:41,12,2019-11
3548,comment,JMU is going to go online and you all should p...,"I agree, wholeheartedly. Many of my professors...",2020-07-22 12:32:05,3,2020-07


## 📖 2 Follow Along — Split `text` into sentences

You do not need to write or modify any code in this section. Run each cell and focus on understanding what the code is doing and why.

To make our lives a bit easier, we will first split each post into individual sentences. There are two reasons for this:
1. Tokenizers are quicker when they work on short text.
2. We are looking at the emotions around a place-per-sentence. 

The cell below does two things in sequence:

- **`.apply(nltk.sent_tokenize)`** — runs the sentence splitter on every row in the `text` column. Each post becomes a *list* of sentences stored in a new column called `sentences`.
- **`.explode('sentences')`** — takes those lists and gives each sentence its own row. The name sounds dramatic, but all it does is "unpack" the list. A post with 5 sentences becomes 5 rows, each sharing the same metadata (date, score, etc.).

Run the cell to see what the result looks like.

In [4]:
# Step 1: Split each text into individual sentences using NLTK
df_with_sentence_lists = df_reddit.assign(sentences=df_reddit['text'].apply(nltk.sent_tokenize))

# Step 2: Create a new row for each sentence (explode the lists)
df_reddit_sentences = df_with_sentence_lists.explode('sentences')

# Display a random sample of 5 sentences to see the results
df_reddit_sentences.sample(n=5, random_state=43)

,type,title,text,date,score,year_month,sentences
7515,comment,I was just accepted!,"Congratz, you're gonna love it. Pick the quad ...",2014-01-09 17:09:40,8,2014-01,Pick the quad as your first choice for housing
4435,comment,Missing Dog,I don't know how? This dog is now safely home ...,2020-09-16 11:32:26,2,2020-09,This dog is now safely home but would apprecia...
2906,comment,"1 Year Ago, we thought it would only be 2 week...",Don't remind me.,2021-03-11 22:12:23,7,2021-03,Don't remind me.
8214,comment,Its so hard to find friends as an older JMU st...,My daughter went there as transfer student whe...,2025-02-09 21:58:25,2,2025-02,"She ended up joining some ""be kind"" group (som..."
1911,comment,President Alger Mishap!,Holy shit dude I thought I was the only one! M...,2019-08-27 21:36:40,23,2019-08,He just stared at us in the darkness for a goo...


> 📊 **Output:** Each Reddit post has been split into individual sentences, and each sentence now occupies its own row. Notice that `date`, `score`, and `text` all repeat — that data is copied to every sentence that came from the same post. Here is a real example from the dataset where a single post with three sentences becomes three rows:
>
> | date | score | text | sentences |
> |---|---|---|---|
> | 2021-02-04 | 7 | It's been this empty for almost a year now. I'm one of the few people who is there every day. Eerily quiet... | It's been this empty for almost a year now. |
> | 2021-02-04 | 7 | It's been this empty for almost a year now. I'm one of the few people who is there every day. Eerily quiet... | I'm one of the few people who is there every day. |
> | 2021-02-04 | 7 | It's been this empty for almost a year now. I'm one of the few people who is there every day. Eerily quiet... | Eerily quiet... |
>
> The `text` column is now redundant — it's just the `sentences` rows pasted back together. That's why the next step drops it.
> Keep in mind that Python did this for thousands of sentences in a split second. Imagine doing this by hand in a Word doc!

### 2.1 Drop `text` and `title`

Since we are essentially copying `text_data` over and over again it's a good practice to `.drop` it. We already have that information in the new sentences column.

In [5]:
df_reddit_sentences = df_reddit_sentences.drop(columns=['text', 'title'])
df_reddit_sentences.head(5)

,type,date,score,year_month,sentences
0,post,2024-03-18 12:47:10,358,2024-03,President Alger leaving to take same job at Am...
1,comment,2024-03-18 12:49:04,82,2024-03,"Like him or not, he did help transform this sc..."
1,comment,2024-03-18 12:49:04,82,2024-03,Applications to JMU have drastically increased...
2,comment,2024-03-18 12:50:05,34,2024-03,Massive changes happening at JMU this year.
2,comment,2024-03-18 12:50:05,34,2024-03,"Alger stepping down, AD Bourne retiring, Cigne..."


> 📊 **Output:** The table is now more compact — only `date`, `score`, and `sentences` remain. Removing columns that are no longer needed is a standard practice in data work. It keeps the DataFrame readable, reduces memory usage, and makes it harder to accidentally use a column you didn't mean to.

## 3 Filtering by list

The most basic way to figure out if the texts contain places is to go through each text and filter it with a list of known place names. For example, we can create a list of Virginia places:

- 'Richmond'
- 'Harrisonburg'
- 'Lynchburg'
- 'Roanoke'
- 'Charlottesville'


We can then use our `str.contains()` method to filter out any texts that contain the words above.

In [44]:
# Define the Virginia cities we want to search for
virginia_cities = ['Richmond', 'Harrisonburg', 'Lynchburg', 'Roanoke', 'Charlottesville']

# Filter sentences that contain any of these city names
# The '|' symbol means "OR" - so we're looking for sentences with ANY of these cities
df_reddit_words = df_reddit_sentences[df_reddit_sentences.sentences.str.contains('|'.join(virginia_cities))]

# Some advanced formatting to show two important examples in the result.
row_1114 = df_reddit_words["sentences"].loc[[1114]]
row_9367 = (
    df_reddit_words["sentences"].loc[[9367]].iloc[[3]]
)  # 4th sentence from this post

(
    pd.concat([row_1114, row_9367])
    .to_frame()
    .reset_index()
    .rename(columns={"index": "record"})
    .style.set_properties(**{"text-align": "left", "white-space": "normal"})
)

,record,sentences
0,1114,"I am an alum and during my time I have come to love the students, staff, and professors (honestly most of my professors were pretty solid) of JMU and the Harrisonburg +valley community."
1,9367,There is no direct bus that goes from Hburg to Lynchburg so he had to pick me up from Lex to drive me to his house.


> 📊 **Output:** These two records expose the core weakness of searching by a fixed word list — the problem of the **unknown unknown**.
>
> - **Record 1114** contains "the Valley," a common local shorthand for the Shenandoah Valley. No official gazetteer will list "the Valley" as a place name, so it would be invisible to any list-based search, no matter how exhaustive.
> - **Record 9367** mentions "Hburg" and "Lex" — casual abbreviations for Harrisonburg and Lexington that people use in everyday writing. We searched for "Harrisonburg," so we caught this sentence, but we never considered that students might write "Hburg" instead. And Lexington was not on our list at all.
>
> The deeper issue is that you cannot search for what you do not know exists. A list only finds what you already thought to include. Named Entity Recognition, covered in the next section, sidesteps this entirely: rather than matching against a predefined set of strings, it reads the grammar of each sentence and identifies place names on its own — including abbreviations, nicknames, and places you never thought to look for.

## 4 Using Named Entity Recognition

Working with location names represents a unique computational problem. Since we do not know in advance what the names might be, and since some proper nouns can be both cities and people, (i.e. Jefferson, Washington, Lincoln, etc.) we need the computer to have some concept of what a place is. This is where language models come in. 

Language models are a form of **Machine Learning (ML)** — a branch of Artificial Intelligence (AI) in which a system learns patterns from large amounts of data rather than following hand-written rules. They have been trained on a lot of text. Through statistical inference they establish the different types of **entities** or words in a text. These can be **parts of speech** like a verb, noun, adjective etc. With this basic understanding of grammar, they can infer more complex **entities** people, places, and organizations. 

One of the main libraries that Python uses to do this is `sPacy`. The sample below goes through the basic procedure for extracting an entity. 

**Don't worry about how the code works for now, just look at the result**

The code passes through the sentence:
>The University of Virginia is in the town of Charlottesville. 
          It was created by Jefferson. In the 1950s, William Faulkner gave a series of lectures there 
          about his fiction, most of which is set in Jefferson.



In [51]:
import spacy

# Load the small English model — "sm" = small, fast, good enough for demos
nlp = spacy.load('en_core_web_sm')

# Our example sentence — deliberately contains a person named Jefferson AND
# a place named Jefferson so we can see whether spaCy tells them apart
text = """The University of Virginia is in the town of Charlottesville. 
          It was created by Jefferson. In the 1950s, William Faulkner gave a series of lectures there 
          about his fiction, most of which is set in Jefferson."""

# nlp() runs the full pipeline: tokenize → tag parts of speech → find entities
doc = nlp(text)

# This coding helps display the result in a human-readable format. Understanding how it works is not important.
print(f"Text:\n{text.strip()}\n")
print("Named entities:")
for ent in doc.ents:
    print(f"  {ent.text!r:30} {ent.label_!r:10} {spacy.explain(ent.label_)}")

Text:
The University of Virginia is in the town of Charlottesville. 
          It was created by Jefferson. In the 1950s, William Faulkner gave a series of lectures there 
          about his fiction, most of which is set in Jefferson.

Named entities:
  'The University of Virginia'   'ORG'      Companies, agencies, institutions, etc.
  'Charlottesville'              'GPE'      Countries, cities, states
  'Jefferson'                    'PERSON'   People, including fictional
  'the 1950s'                    'DATE'     Absolute or relative dates or periods
  'William Faulkner'             'PERSON'   People, including fictional
  'Jefferson'                    'GPE'      Countries, cities, states



> 📊 **Output:** Notice how accurately spaCy is able to distinguish between an organization in Virginia (UVA), a person Jefferson, and the place Jefferson (geopolitical entity).

spaCy is only as accurate as the data provided to it. If the text data is garbled or too short, it will likely have trouble. Undoubtedly, there are sentences in our `sentences` column that are not going to be read properly, but what we are relying on is the sheer volume of text. Even with some false positives and false negatives, we should be able to build a pretty good overview of the most mentioned places.

## 5 Extract Entities in all `sentences`

Doing one extraction on one sentence in `spacy` is pretty straight forward. We simply run the function `nlp()` on whatever sentence we want to analyze and save the result to a new variable, usually called `doc`. When we run this on a column with thousands of sentences, you start to run into performance issues because you are doing the procedure one at a time, and you also don't really know what's going on because there's no feedback. The functions below modify the above procedure a bit and basically asks your computer to use multiple processors, and it also provides a little progress bar. Finally, instead of using the very small model that we used above, we are now going to use a slightly bigger model called `en_core_web_md`, this will hopefully help us find more locations!

We are now ready to apply this function to `sentences` and create a new column called `toponyms`. 

**Warning this process will take a couple of minutes**

If this does not work, I have saved the results as `jmu_reddit_toponyms.pickle` and preloaded it in your data folder. You can simply keep running the code below on that imported file.

In [54]:
import spacy
from tqdm import tqdm

# Load the medium English model — more accurate than "sm" for location detection
nlp = spacy.load('en_core_web_md')

# nlp.pipe() processes all sentences in batches — much faster than one at a time.
# batch_size=256 means 256 sentences are grouped together per pass.
# tqdm displays a progress bar so you can track how far along the process is.
sentences = df_reddit_sentences['sentences']
toponyms = []

for doc in tqdm(nlp.pipe(sentences, batch_size=256), total=len(sentences)):
    # For each sentence, collect any entities labelled GPE (Geopolitical Entity = places)
    gpes = [ent.text for ent in doc.ents if ent.label_ == 'GPE']
    toponyms.append(gpes if gpes else None)

# Store the results as a new column in the DataFrame
df_reddit_sentences['toponyms'] = toponyms

print(f"✅ Done — processed {len(sentences):,} sentences")

✅ Loaded spaCy model: en_core_web_md


100%|██████████| 30005/30005 [01:13<00:00, 409.00it/s]


> 📊 **Output**: You should see around 30,000 sentences processed in about 1–3 minutes. Keep in mind that this is an incredibly complex task for a computer. For every single sentence it has to: read the text, parse the grammar, decide whether any words are locations, extract those locations, and store the result — then repeat the whole process for the next sentence. The fact that it completes ~30,000 of these analyses in a few minutes is a remarkable feat of modern NLP. A human researcher doing this manually would need weeks.

### Analyze the results

Run the code below to show the table and the results. The display has been separated from the processing because you do not want to process the data every time you want to view the results. 

In [120]:
(
    df_reddit_sentences[['sentences', 'toponyms']]
    .sample(5, random_state=109)
    .style.set_properties(**{"text-align": "left", "white-space": "normal"})
)

,sentences,toponyms
9215,"i eventually DID find a roommate and we get along extraordinarily well, but if you’d rather have a room by yourself or with someone better aligned to your identity- that’s something especially available as of recent.",None
4600,"Former Chesapeake resident, TIL Potomac is now Chandler.","['Chesapeake', 'TIL Potomac']"
10721,"I worked at the mall and during the winter my Soph and Senior years I worked at Massanutten and Wintergreen, mostly so I could ski for free and make a little extra money.",['Massanutten']
9303,Lots of group projects.,None
11318,"I already had it make my phone freeze after trying to verify, and then it wouldn't accept one verification so I had to say yes multiple times",None


> 💡 **Reflection:** Look at the locations `spaCy` found in this sample. Which ones could you have found with a simple keyword list — city names you already knew to search for? Which ones might you never have thought to include? And are there any sentences that clearly mention a place that `spaCy` missed entirely? Keep those examples in mind: they are the evidence for why NER is more powerful than a list, but also why it is not perfect.

Because not every sentence includes a toponym sometimes it will say `None`. We want to eliminate these rows because they are not relevant. Still, we might want to peek inside and calculate what percentage of sentences actually have toponyms. The calculation below achieves exactly this. It creates a table of the number of sentences with toponyms, and then divides the number of rows in that table by the total number of rows in the data set. This gives the percentage of rows that contain locations. In this case, around 4%.

In [128]:
# Filter to sentences that have at least one toponym — None rows are dropped here
df_reddit_toponyms = df_reddit_sentences[df_reddit_sentences['toponyms'].notna()]
total = len(df_reddit_sentences)

print(f"Sentences with at least one toponym: {len(df_reddit_toponyms):,} of {total:,} ({len(df_reddit_toponyms) / total * 100:.1f}%)")

Sentences with at least one toponym: 1,217 of 30,005 (4.1%)


> 💡 **Reflection:** Only about 4% of sentences contain a place name. This is a pretty low number. What does that tell you about how people write on Reddit? Are most posts about events and opinions rather than places? If we are only using a small subset of sentences, how might that distort our results when we look at sentiment by locations?


## 6 Counting Toponyms

The code below counts how often each place name appears across the entire dataset. It does this in two steps:

1. **Flatten** — each row currently holds a *list* of toponyms (a sentence with two places has two items in its list). `.explode()` unpacks those lists so each toponym gets its own row, the same operation we used in Section 2.
2. **Count** — `.value_counts()` counts how many times each toponym appears and sorts the result from most to least common.

In [ ]:
# Step 1: Flatten the lists — each toponym gets its own row (same .explode() we used earlier)
unnested = df_reddit_toponyms['toponyms'].explode()

# Step 2: Count and sort — .value_counts() counts occurrences, most common first
toponym_counts_df = (
    unnested
    .value_counts()
    .reset_index()
    .rename(columns={'toponyms': 'Toponym', 'count': 'Count'})
)

toponym_counts_df.head(10)

,Toponym,Count
4,Harrisonburg,240
7,Virginia,114
8,VT,47
40,VA,42
19,US,41
72,Florida,21
66,harrisonburg,20
23,America,19
65,Breeze,19
47,FCS,19


> 💡 **Reflection:** Not surprisingly, Harrisonburg is the top toponym. But look further down the list — there are also entries like `FCS` and `Breeze` that are not places at all. What other results look like odd to you? What might need to be fixed down the road? 

### 7 Visualizing Toponyms

A bar chart is a natural fit here — each bar represents one place name, and its height shows how often it appears. `px.bar()` takes the DataFrame we just built and maps `Toponym` to the x-axis and `Count` to the y-axis.

In [132]:
import plotly.express as px

# Take the top 10 most common toponyms for plotting
toponym_counts_top10 = toponym_counts_df.head(10)

# Create the bar chart using Plotly
fig = px.bar(
    toponym_counts_top10,
    x='Toponym',
    y='Count',
    title='Top 10 Most Common Toponyms',
    text='Count'
)

# Display the plot
fig.show()

> 💡 **Reflection:** Take a look at some of the names that are synonyms. If you were to stack these bars up on top of each other, what would the chart look like? What three places dominate discussion on the reddit thread?

### 6.2 Engagement by Toponym

Raw counts tell us what places are *mentioned* most often. But do those same places also generate the most discussion — the posts that get the most upvotes?

To answer that, we need two numbers per place at the same time: how often it appears, and the average score of posts that mention it. The code below computes both in a single step using `.groupby().agg()`:

| Step | What it does |
|---|---|
| **`.explode()`** | Same as before — one toponym per row |
| **`.groupby('toponyms')`** | Group all rows that share the same place name |
| **`.agg(...)`** | For each group, compute multiple summary statistics at once — here, `count` and `mean` |

In [142]:
# Flatten: one toponym per row, keep the post score alongside it
toponym_score_df = (
    df_reddit_toponyms[['toponyms', 'score']]
    .explode('toponyms')
    .dropna(subset=['toponyms'])
)

# Group by toponym: count mentions AND average the post score
# Filter to places mentioned at least 10 times — rare mentions can have
# artificially high average scores, so a minimum count gives cleaner results
toponym_engagement = (
    toponym_score_df
    .groupby('toponyms', as_index=False)
    .agg(Count=('toponyms', 'count'), Avg_Score=('score', 'mean'))
    .rename(columns={'toponyms': 'Toponym'})
    .query('Count >= 10')
    .sort_values(['Avg_Score', 'Count'], ascending=False)
    .reset_index(drop=True)
)

toponym_engagement.head(15)

,Toponym,Count,Avg_Score
0,US,41,25.390244
1,Virginia,114,23.508772
2,Breeze,19,22.789474
3,Harrisonburg,240,20.316667
4,harrisonburg,20,17.700000
5,Florida,21,17.095238
6,VT,47,17.021277
7,Richmond,15,15.066667
8,Jersey,17,14.000000
9,VA,42,13.404762


In [146]:
# Take the top 30 by engagement for plotting
top30_engagement = toponym_engagement.head(30)

# Treemap: each box is one place name
#   Box SIZE  → Count        (bigger box = mentioned more often)
#   Box COLOR → Avg_Score    (darker blue = higher average upvote score)
fig = px.treemap(
    top30_engagement,
    path=['Toponym'],
    values='Count',
    color='Avg_Score',
    title='Toponyms: Box Size = Mentions · Color = Average Upvote Score',
    labels={'Avg_Score': 'Avg Score', 'Count': 'Mentions'},
    color_continuous_scale='Blues',
)

fig.update_traces(textinfo='label+value')
fig.show()

> 💡 **Reflection:** In this chart, box size shows how often a place was mentioned and color shows how much engagement those posts generated. Which places are both large *and* dark — frequently mentioned and highly engaging? Are there any places with a small box but a deep color, meaning they come up rarely but generate a strong reaction when they do? What might explain that pattern?

## 7 Save & Export

We need to save our work in two ways for two different purposes:

- **Pickle** — preserves the exact pandas data types we carefully set in Lesson 3 (category, StringDtype, datetime). This is what the next Python notebook will load.
- **CSV** — a plain text format that Google Sheets can open. This is what your team will use for the collaborative review step.

Both files go in the `data/` folder.

In [ ]:
# Save the filtered DataFrame (sentences that have toponyms) as a pickle.
# This preserves dtypes so the next notebook doesn't have to re-cast everything.
df_reddit_toponyms.to_pickle('data/jmu_reddit_toponyms.pickle')
print("✅ Saved data/jmu_reddit_toponyms.pickle")

### 7.1 Prepare for Google Sheets Export

Before we export, we need to add two things:

1. A **`unique_id`** — a stable number for each row. This is critical because after your team edits the file in Google Sheets and re-downloads it, we need to be able to reconnect each reviewed row back to the original sentence and its score. Without this anchor the data can't be rejoined.

2. A **`school_name`** column — in this lesson we only have JMU data, but the full analysis compares multiple Virginia universities. Adding this column now means the exported format is consistent.

We also need to **explode** the `toponyms` column. Right now each row holds a *list* of places found in one sentence. Google Sheets can't work with lists in a cell, so we split it so that each place gets its own row.

In [ ]:
# Assign stable sequential IDs before any further filtering
df_reddit_toponyms = df_reddit_toponyms.reset_index(drop=True)
df_reddit_toponyms.insert(0, 'unique_id', df_reddit_toponyms.index)

# Add school_name — in the multi-school version this column already exists in the source CSV
if 'school_name' not in df_reddit_toponyms.columns:
    df_reddit_toponyms['school_name'] = 'JMU'

# Explode: one row per (sentence, extracted_place) pair
df_locations_raw = (
    df_reddit_toponyms[['unique_id', 'school_name', 'date', 'score', 'sentences', 'toponyms']]
    .explode('toponyms')
    .rename(columns={'sentences': 'sentence', 'toponyms': 'extracted_place'})
    .dropna(subset=['extracted_place'])
    .reset_index(drop=True)
)

df_locations_raw.head(10)

In [ ]:
# Export to CSV for Google Sheets
df_locations_raw.to_csv('data/locations_raw.csv', index=False)

print(f"✅ Saved data/locations_raw.csv")
print(f"   Rows (one per sentence-place pair): {len(df_locations_raw):,}")
print(f"   Unique extracted place names:        {df_locations_raw['extracted_place'].nunique():,}")
print(f"   Columns: {list(df_locations_raw.columns)}")

---

## 8 Google Sheets — Collaborative Review

**Stop here. Before running any more code, your team needs to review the data.**

You've just seen that spaCy found a lot of places — but also a lot of noise. `FCS` is not a place. `Breeze` (JMU's student newspaper) is not a place. And "Dining Hall" is a real place on campus, but it's so generic that a geocoder will have no idea what coordinates to assign to it.

This is where your team comes in.

### What to do

1. **Upload `data/locations_raw.csv` to Google Sheets.** You can do this by going to [sheets.google.com](https://sheets.google.com), creating a new spreadsheet, and importing the CSV file.

2. **Add these four columns** to the right of the existing ones (exact spellings matter — Python will look for them):

   | Column | What to put in it |
   |---|---|
   | `confirmed_place` | The real place name. Copy from `extracted_place` if it's correct. Write a better name if it's vague (e.g. "Dining Hall" → "Festival Dining Hall, JMU"). **Leave blank** if it's a false positive. |
   | `confirmed_lat` | Latitude. Only needed for campus-specific places a geocoder won't know (e.g. `38.4332`). Leave blank for real cities — the geocoder will find those. |
   | `confirmed_lon` | Longitude. Same rule as above (e.g. `-78.8703`). |
   | `is_false_positive` | `TRUE` if this is not actually a place (e.g. `FCS`, `Breeze`, `The`). `FALSE` otherwise. |
   | `checked_by` | Your name, so we know who reviewed each row. |

3. **Divide the rows** among your team so each person reviews a section. You can filter by `extracted_place` to group similar names together.

4. **Download the finished sheet** as a CSV file. In Google Sheets: `File → Download → Comma-separated values (.csv)`. Save it as `locations_cleaned.csv` inside the `data/` folder in this project.

5. **Come back and run the cells below.**

---

> 💡 **Tips for reviewing:**
> - If you're unsure whether something is a real place, search for it in Google Maps.
> - For campus buildings, search `"[building name] JMU"` in Google Maps to get the exact coordinates.
> - Be consistent: if one person writes `"Harrisonburg, VA"` and another writes `"Harrisonburg"`, they will count as different places in the analysis. Agree on a naming convention with your team.

## 9 Re-import the Cleaned Data

Welcome back. Let's load your team's reviewed file and see what changed.

In [ ]:
# Load the cleaned file your team produced in Google Sheets.
# If your team hasn't finished yet, a pre-reviewed version is available as a fallback:
# df_locations_cleaned = pd.read_csv('data/locations_cleaned_backup.csv')

df_locations_cleaned = pd.read_csv('data/locations_cleaned.csv')

print(f"Loaded {len(df_locations_cleaned):,} rows")
print(f"Columns: {list(df_locations_cleaned.columns)}")
df_locations_cleaned.head(10)

### 9.1 What Did We Change?

Let's measure how much the human review improved the data. This kind of audit is standard practice in computational humanities — it's how you justify your results.

In [ ]:
total = len(df_locations_cleaned)
false_positives = df_locations_cleaned['is_false_positive'].sum()
corrected = (
    df_locations_cleaned['confirmed_place'].notna() &
    (df_locations_cleaned['confirmed_place'] != df_locations_cleaned['extracted_place'])
).sum()
manually_geocoded = df_locations_cleaned['confirmed_lat'].notna().sum()

print("── Review Summary ─────────────────────────────────")
print(f"  Total rows reviewed:              {total:,}")
print(f"  False positives removed:          {false_positives:,}  ({false_positives/total*100:.1f}%)")
print(f"  Place names corrected/clarified:  {corrected:,}  ({corrected/total*100:.1f}%)")
print(f"  Manually geocoded (campus places):{manually_geocoded:,}  ({manually_geocoded/total*100:.1f}%)")
print("────────────────────────────────────────────────────")

In [ ]:
# Show examples of corrections your team made
corrections = df_locations_cleaned[
    df_locations_cleaned['confirmed_place'].notna() &
    (df_locations_cleaned['confirmed_place'] != df_locations_cleaned['extracted_place'])
][['extracted_place', 'confirmed_place', 'confirmed_lat', 'confirmed_lon', 'checked_by']].drop_duplicates('extracted_place')

print(f"Sample corrections ({min(15, len(corrections))} of {len(corrections)}):")
corrections.head(15)

### 9.2 Save the Cleaned Data

The cleaned file is now ready to hand off to the geoparsing step in the next notebook. We save it as both pickle (for dtypes) and CSV (for transparency).

In [ ]:
# Remove false positives before saving — they're done with
df_locations_verified = df_locations_cleaned[~df_locations_cleaned['is_false_positive']].copy()

# Pickle: preserves dtypes for the next notebook
df_locations_verified.to_pickle('data/locations_verified.pickle')

# CSV: human-readable audit trail
df_locations_verified.to_csv('data/locations_verified.csv', index=False)

print(f"✅ Saved data/locations_verified.pickle  ({len(df_locations_verified):,} rows)")
print(f"✅ Saved data/locations_verified.csv     ({len(df_locations_verified):,} rows)")
print(f"\nThese files are the input for Lesson 4.2 (geoparsing).")